# Historical baseline notebook

This records the initial MSE-based baseline experiment. MSE is not the competition ranking metric. For the later official-score model selection, see `01_walkforward_research.ipynb`. Repeatedly inspected quarters are development validation, not an untouched test set.


# 我的第一版策略：财报文本 → 收益百分位

**TF-IDF + Ridge · 无需 OpenAI key · 只做离线研究**

把本文件放在 `~/explaining-markets-research/notebooks/`，选择该项目 `.venv`，从上往下运行。
先在终端安装：`python -m pip install scikit-learn joblib`。
需要先运行官方 notebook，将完整档案缓存到 `data/archive/`。本 notebook 不访问 API、不打印密钥、不提交预测、不修改已部署服务。

输入只使用官方 `earnings-call-facts` 财报事实。TF-IDF 把词与双词组合转换成数值；Ridge 学习带 L2 正则化的线性关系。
目标是每季度内 `car1` 的百分位，不是上涨概率，也不是原始收益百分比。

流程：检查数据 → 较早季度训练 → 倒数第二个季度选 alpha → 最后一个已结束季度测试。
当前或未来季度一律排除；每季度标签先在有效收益的完整横截面排名，再处理文本缺失。
文本缺失的测试记录仍计入评估，使用固定 0.5。

**解释范围**：这是事后档案的时间留出测试。披露文本、收益可能事后生成或修订；没有历史版本快照，不能证明当时可获得这些材料/标签。
季度结束不代表档案已封存；代码会记录数据文件哈希以便复现。正式实盘还需核对信息截止时间、标签可得时间、材料版本和事件集合。
MSE、Spearman、预测 R² 是诊断指标，**不是官方排行榜分数**。
标签使用本地官方 `examples.scoring.percentile_ranks`；最终排行榜还涉及额外评分规则与共同事件集合。


In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from examples.archive import load_archive
from examples.summary import facts_from_disclosure
from examples.scoring import percentile_ranks

roots = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in roots if (p / 'data' / 'archive').is_dir()), None)
if REPO is None:
    raise RuntimeError('请把 notebook 放在研究项目 notebooks 文件夹，并先下载完整历史档案。')
ARCHIVE_DIR = REPO / 'data' / 'archive'
archive_files = sorted(ARCHIVE_DIR.rglob('*.jsonl.gz'))
if not archive_files:
    raise RuntimeError('data/archive 中没有档案，请先运行官方历史数据 notebook 的下载步骤。')
df = load_archive(ARCHIVE_DIR)
print('读取档案事件数：', len(df))
display(df.groupby(['quarter', 'return_status'], dropna=False).size().unstack(fill_value=0))


## 1. 清理与构造答案

一行对应一个事件中的一个 focal asset。只读取 finite 且状态为 ok 的收益；保留文本缺失样本以避免隐藏覆盖问题。
同一事件/股票如出现重复，要求记录一致，否则停止并检查，不随意选择一个版本。
`metrics`、`baseline_predictions`、`event_returns` 都不会进入文本特征。


In [ ]:
def facts_text(record):
    disclosure = record.get('disclosure')
    if not isinstance(disclosure, dict):
        return ''
    facts = facts_from_disclosure(record)
    return '\n'.join(f.strip() for f in facts if isinstance(f, str) and f.strip())

def make_outcomes(frame):
    rows, skipped = [], Counter()
    for rec in frame.to_dict('records'):
        if rec.get('event_type') != 'EARNINGS_RELEASE':
            skipped['not_earnings'] += 1
            continue
        if not isinstance(rec.get('event_id'), str):
            skipped['missing_event_id'] += 1
            continue
        quarter = rec.get('quarter')
        if not isinstance(quarter, str) or not re.fullmatch(r'\d{4}Q[1-4]', quarter):
            skipped['invalid_quarter'] += 1
            continue
        assets, returns = rec.get('focal_assets'), rec.get('event_returns')
        if not isinstance(assets, list) or not isinstance(returns, dict):
            skipped['missing_assets_or_returns'] += 1
            continue
        text = facts_text(rec)
        for asset in assets:
            if not isinstance(asset, dict):
                continue
            ticker = asset.get('identifier_value')
            if not isinstance(ticker, str) or not ticker:
                continue
            leg = returns.get(ticker)
            if not isinstance(leg, dict) or leg.get('return_status') != 'ok':
                skipped['invalid_return_status'] += 1
                continue
            value = leg.get('car1')
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
                skipped['invalid_car1'] += 1
                continue
            rows.append(dict(event_id=rec['event_id'], ticker=ticker, quarter=quarter,
                             event_datetime=rec.get('event_datetime'), text=text, car1=float(value)))
    if not rows:
        raise RuntimeError('没有找到有效财报收益，请检查是否使用了完整档案。')
    out = pd.DataFrame(rows)
    keys = ['event_id', 'ticker']
    duplicates = out[out.duplicated(keys, keep=False)]
    if not duplicates.empty:
        conflict = duplicates.groupby(keys)[['quarter', 'car1', 'text', 'event_datetime']].nunique(dropna=False)
        if (conflict > 1).any().any():
            raise RuntimeError('重复事件记录存在冲突，请检查档案版本。')
        out = out.drop_duplicates(keys).copy()
    out['event_datetime'] = pd.to_datetime(out['event_datetime'], utc=True, errors='coerce')
    if out['event_datetime'].isna().any():
        raise RuntimeError('存在无效事件时间，请先检查。')
    # 官方每季度百分位变换；不是 pandas rank(pct=True)。
    out['y'] = out.groupby('quarter')['car1'].transform(lambda s: percentile_ranks(s.tolist()))
    out['has_text'] = out['text'].str.contains(r'[A-Za-z]{2}', regex=True)
    return out.sort_values(['event_datetime', 'event_id', 'ticker']).reset_index(drop=True), skipped

data, skipped = make_outcomes(df)
print('跳过记录/收益原因（单位可能不同）：', dict(skipped))
print('有效事件-股票行数：', len(data))
display(data.groupby('quarter').agg(outcomes=('y','size'), with_text=('has_text','sum'),
                                  start=('event_datetime','min'), end=('event_datetime','max')))


## 2. 按季度划分，保留最后一个季度做测试

自动使用已结束的季度。2026 年 11 月运行时排除 2026Q4。
至少需要 3 个已结束季度：早期用于训练，倒数第二个用于验证，最后一个用于测试。
如果不足，停止，不偷偷改成随机划分。不要反复查看测试结果后调参，否则它也变成了验证集。


In [ ]:
now = datetime.now(timezone.utc)
current_quarter = f'{now.year}Q{(now.month-1)//3+1}'
quarters = sorted(q for q in data['quarter'].unique() if q < current_quarter)
if len(quarters) < 3:
    raise RuntimeError(f'只有 {quarters} 这些已结束季度。需要至少 3 个季度，请先检查档案。')
train_quarters, valid_quarter, test_quarter = quarters[:-2], quarters[-2], quarters[-1]
train = data[data['quarter'].isin(train_quarters)].copy()
valid = data[data['quarter'].eq(valid_quarter)].copy()
test = data[data['quarter'].eq(test_quarter)].copy()
assert train['event_datetime'].max() < valid['event_datetime'].min()
assert valid['event_datetime'].max() < test['event_datetime'].min()
assert set(train['event_id']).isdisjoint(valid['event_id'])
assert set(train['event_id']).isdisjoint(test['event_id'])
assert set(valid['event_id']).isdisjoint(test['event_id'])
for name, part in [('train',train), ('validation',valid), ('test',test)]:
    if len(part) < 30 or part['has_text'].sum() < 20:
        raise RuntimeError(f'{name} 样本或文本太少，不适合这套测试：{len(part)} 行。')
    print(name, sorted(part['quarter'].unique()), '总行数',len(part), '有文本',int(part['has_text'].sum()))
print('排除当前/未来季度：', sorted(set(data['quarter']) - set(quarters)))


## 3. 训练与选正则化强度 alpha

TF-IDF 的词表只在训练数据上拟合，验证/测试文本不会用来建立词表。
用验证集 MSE 选择 alpha。alpha 越大，系数受到的约束越强。
只用财报事实，保留否定词，不使用英文停用词列表。


In [ ]:
def new_model(alpha):
    return Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2,
                                max_features=30000, sublinear_tf=True, dtype=np.float64)),
        ('ridge', Ridge(alpha=alpha, solver='lsqr')),
    ])

def predict_frame(model, part):
    pred = np.full(len(part), 0.5)
    mask = part['has_text'].to_numpy()
    if mask.any():
        pred[mask] = np.clip(model.predict(part.loc[mask, 'text']), 0.0, 1.0)
    return pred

def diagnostic_metrics(y, pred):
    y, pred = np.asarray(y, float), np.asarray(pred, float)
    correlation = pd.Series(y).corr(pd.Series(pred), method='spearman') if np.std(pred) > 1e-12 else np.nan
    return dict(MSE=mean_squared_error(y,pred), Spearman=correlation, prediction_R2=r2_score(y,pred))

fit_train = train[train['has_text']]
validation_rows = []
for alpha in [0.1, 1.0, 10.0, 100.0]:
    candidate = new_model(alpha)
    candidate.fit(fit_train['text'], fit_train['y'])
    pred = predict_frame(candidate, valid)
    validation_rows.append(dict(alpha=alpha, **diagnostic_metrics(valid['y'],pred)))
validation_results = pd.DataFrame(validation_rows).sort_values('MSE').reset_index(drop=True)
display(validation_results)
best_alpha = float(validation_results.iloc[0]['alpha'])
print('选定 alpha：', best_alpha)
print('验证集固定 0.5 的 MSE：', mean_squared_error(valid['y'], np.full(len(valid),0.5)))


## 4. 固定参数，训练 + 验证合并重训，再评估一次测试集

这一步之后不要根据测试季度调整 alpha。若要继续调参，需要新的独立留出数据。
`prediction_R2` 是直接预测的样本外 R²（可能为负），不是官方的回归评分 R²。
固定 0.5 没有排序能力，因此它的 Spearman 显示 NaN 是正常的。


In [ ]:
development = pd.concat([train,valid], ignore_index=True)
fit_development = development[development['has_text']]
model = new_model(best_alpha)
model.fit(fit_development['text'], fit_development['y'])
test_predictions = predict_frame(model,test)
constant_predictions = np.full(len(test),0.5)
results = pd.DataFrame([
    dict(model='Constant 0.5', **diagnostic_metrics(test['y'],constant_predictions)),
    dict(model='TF-IDF + Ridge', **diagnostic_metrics(test['y'],test_predictions)),
]).set_index('model')
display(results)
gain = 1 - results.loc['TF-IDF + Ridge','MSE']/results.loc['Constant 0.5','MSE']
print(f'测试季度 {test_quarter}；相对固定 0.5 的 MSE 改善：{gain:.2%}（负数代表更差）')
print(f'文本覆盖率：{test.has_text.mean():.1%}；预测标准差：{test_predictions.std():.4f}')
test_output = test[['event_id','ticker','quarter','event_datetime','car1','y','has_text']].copy()
test_output['prediction'] = test_predictions
display(test_output.head(10))


## 5. 看误差与文本系数

按预测值分组，检查预测较高的组是否真的有更高的平均答案。
系数表只说明此模型在当前词表中的关联，不能解释为因果，也不是单个词的独立投资信号。


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].hist(test_predictions,bins=25,color='#3267ad')
axes[0].axvline(.5,color='gray',linestyle='--')
axes[0].set(xlabel='Predicted percentile',ylabel='Count',title=f'Prediction distribution: {test_quarter}')
bins = test_output.copy()
bins['bin'] = pd.qcut(bins['prediction'],q=10,duplicates='drop')
calibration = bins.groupby('bin',observed=True).agg(prediction=('prediction','mean'), actual=('y','mean'), n=('y','size'))
if not calibration.empty:
    axes[1].plot(calibration['prediction'],calibration['actual'],'o-',color='#3267ad')
axes[1].plot([0,1],[0,1],'--',color='gray')
axes[1].set(xlabel='Mean prediction',ylabel='Mean realized percentile',title='Binned calibration',xlim=(0,1),ylim=(0,1))
plt.tight_layout()
plt.show()
display(calibration)
coefficients = pd.DataFrame({'term':model['tfidf'].get_feature_names_out(),'coefficient':model['ridge'].coef_})
display(coefficients.nsmallest(12,'coefficient').reset_index(drop=True))
display(coefficients.nlargest(12,'coefficient').reset_index(drop=True))


## 6. 保存研究结果（默认不保存）

确认结果后将 `SAVE_RESULTS` 改成 `True`。保存的是只训练到测试季度之前的模型；不会自动部署。
模型只接受相同格式的财报事实文本，部署时还需要统一提取函数、配置依赖、上传模型、测试与重新部署。
如果要训练供未来事件使用的版本，可以之后再使用已经揭晓的测试季度；旧测试成绩不能用来评估那个重训模型。


In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    destination = REPO / 'artifacts' / ('tfidf_ridge_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    destination.mkdir(parents=True,exist_ok=False)
    file_hashes = {}
    for path in archive_files:
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda:stream.read(1024*1024),b''):
                digest.update(block)
        file_hashes[str(path.relative_to(REPO))] = digest.hexdigest()
    metadata = dict(strategy='facts_tfidf_ridge_v1', sklearn_version=sklearn.__version__,
                    alpha=best_alpha, train_quarters=train_quarters, validation_quarter=valid_quarter,
                    test_quarter=test_quarter, label='official percentile_ranks(car1) within each quarter',
                    fit_quarters=sorted(development.quarter.unique().tolist()),
                    input='earnings-call-facts text only', missing_text_prediction=0.5,
                    archive_sha256=file_hashes, backtest_type='retrospective archive temporal holdout; not point-in-time verified')
    joblib.dump(model,destination/'model.joblib')
    test_output.to_csv(destination/'test_predictions.csv',index=False)
    validation_results.to_csv(destination/'validation_metrics.csv',index=False)
    results.to_csv(destination/'test_metrics.csv')
    (destination/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
    print('已保存：',destination)
else:
    print('未保存模型；先查看测试结果。没有向比赛平台提交任何预测。')


## 结果怎么读、下一步怎么做

- **MSE 更低**：百分位数值更接近答案；改善小可能只是样本波动，单季度不足以证明稳定有效。
- **Spearman 为正**：预测排序与实际排序正相关；需在更多独立季度检查稳定性。
- **比固定 0.5 更差**：先检查样本和时间覆盖，再只在训练/验证数据上修改，不根据测试集反复试词表。
- 这一版只是文本基准。它不擅长精确比较“实际 EPS vs 预期 EPS”，后续可加入有出处的结构化 surprise / guidance 特征。
- 官方分数应另行用锁定的评分版本、完整共同事件集合和基准 surprise 做评估，不把本页 MSE 当作排行榜成绩。

来源：
- [官方历史数据示例](https://github.com/explaining-markets/examples)
- [规则引用的评分实现](https://github.com/explaining-markets/examples/blob/501bd31/src/examples/scoring.py)
- [比赛规则](https://explainingmarkets.ai/contest-rules)

文件验证范围：生成时以构造数据检查了清理、时间划分、训练、评估与保存流程；未获取你的 10,766 条档案，不含真实回测成绩。
